# Carga y Consolidación de Datos

### Propósito de este notebook

Este notebook implementa el **pipeline de adquisición y consolidación de datos** correspondiente
a la primera etapa de la metodología de investigación. Su función es:

1. Descomprimir los archivos zip bajados, que fueron obtenidos directamente de las fuentes
   Latinobarómetro y V-Dem respectivamente.
2. Leer los 25 archivos `.dta` de Latinobarómetro (1995–2024), extraer las variables
   seleccionadas para la tesis y consolidarlas en un único DataFrame longitudinal.
3. Exportar el DataFrame consolidado como `data/latinobarometro.csv`.
4. Leer el archivo `V-Dem-CY-Core-v16.csv`, filtrar por los países del estudio
   y el período 1995–2024, y exportar el resultado como `data/v-dem.csv`.


### Variables seleccionadas

Las variables fueron definidas y justificadas en la documentación de la tesis.
El mapeo entre los códigos estandarizados del diccionario maestro y los nombres originales
de cada ola se construye automáticamente a partir del archivo Excel de series de tiempo.

---

> **Nota de reproducibilidad:** Todas las semillas aleatorias están fijadas en `42`.
> Las versiones exactas de las librerías utilizadas se encuentran en el archivo requirements.txt.

## 1. Importaciones y configuración global

In [ ]:
# =============================================================================
# Importaciones y configuración global
# =============================================================================

import sys
from pathlib import Path

# Detectar raiz del proyecto (funciona tanto si CWD es notebooks/ como la raiz)
_root = Path(".").resolve()
if not (_root / "utils").exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import os
import gc
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pyreadstat
from tqdm.auto import tqdm
import zipfile

from utils.config import PARAMETERS
from utils.config import PATHS
from utils.config import clean_process_folders

warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


In [ ]:
# =============================================================================
# Variables iniciales y configuración de rutas
# =============================================================================

# --- Verificación de existencia de rutas críticas ----------------------------
rutas_criticas = {
    "Carpeta Latinobarómetro"       : PATHS["FOLDER_RAW_LB"],
    "Carpeta V-Dem"                 : PATHS["FOLDER_RAW_VDEM"],
    "CSV Variables Latinobarómetro" : PATHS["FILE_LB_VAR_MAPPING"]
}

errores = []
for nombre, ruta in rutas_criticas.items():
    estado = "✓" if ruta.exists() else "✗"
    if not ruta.exists():
        errores.append(nombre)
    print(f"  {estado}  {nombre}: {ruta}")

if errores:
    raise FileNotFoundError(
        f"No se encontraron las siguientes rutas: {errores}. "
        "Verificar la estructura de directorios."
    )

print("\n✓ Todas las rutas críticas verificadas correctamente.")

Vaciar carpetas de procesamiento

In [ ]:
clean_process_folders()

## 2. Descomprimir archivos RAW

Se descomprimen los archivos del Latinobarómetros y del V-Dem, 
y se ubican en sus directorios respectivos para que puedan ser procesados.

In [ ]:
# =============================================================================
# Descomprimir archivos ZIP de Latinobarómetro y V-dem
# =============================================================================

for archivo_zip in PATHS["FOLDER_RAW_ZIP"].glob("*.zip"):
    #print(f"Descomprimiendo {archivo_zip.name}...")
    with zipfile.ZipFile(archivo_zip, "r") as zipf:
        if archivo_zip.name.lower().startswith("latinobarometro"):
            zipf.extractall(PATHS["FOLDER_RAW_LB"])
        elif archivo_zip.name.lower().startswith("v-dem"):
            zipf.extractall(PATHS["FOLDER_RAW_VDEM"])
        else:
            print(f"Archivo ZIP no reconocido: {archivo_zip.name}. Se omite la extracción")

## 3. Construcción del diccionario de mapeo variable ↔ ola del Latinobarómetro

El diccionario de mapeo traduce los **códigos estandarizados** del diccionario maestro
(ej. `A_003_031`) a los **nombres originales de columna** de cada archivo `.dta`
(ej. `p21` en LAT1995, `P11ST.A` en LAT2010).  

Este mapeo se construye automáticamente desde el archivo "latinobarometro_variables.csv", 
el cual se basa en el Excel de series de tiempo,
garantizando que cualquier actualización del csv de variables se refleje
automáticamente sin necesidad de modificar el código.

**Nota**: Para cada nueva ola que se incorpore en el Latinobarómetro es necesaria actualizar el archivo csv de variables.

In [ ]:
# =============================================================================
# Construcción del diccionario de mapeo desde el csv de variables del Latinobarómetro
# =============================================================================

# Variables seleccionadas para la tesis (códigos estandarizados del diccionario maestro)
# Organizadas por bloque temático según la metodología de investigación
VARIABLES_SELECCIONADAS = [
    # --- Variable dependiente ---
    "A_003_031",   # Satisfacción con la democracia (TARGET)

    # --- Apoyo al régimen democrático ---
    "A_001_001",   # Apoyo a la democracia
    "A_003_021",   # Escala de desarrollo de democracia

    # --- Confianza institucional ---
    "H_002_011",   # Confianza: Congreso
    "H_002_031",   # Confianza: Gobierno
    "H_002_041",   # Confianza: Poder Judicial
    "H_002_101",   # Confianza: Iglesia
    "H_002_111",   # Confianza: Policía
    "H_002_131",   # Confianza: Televisión
    "H_002_161",   # Confianza: Fuerzas Armadas
    "H_002_241",   # Confianza: Partidos Políticos
    "H_001_011",   # Confianza interpersonal

    # --- Evaluación económica ---
    "D_001_001",   # Situación económica actual del país
    "D_001_021",   # Situación económica del país vs. hace 12 meses
    "D_001_041",   # Situación económica futura del país
    "D_001_061",   # Situación económica personal actual
    "D_001_091",   # Situación económica futura personal
    "D_001_131",   # Ingreso subjetivo
    "C_003_003_011", # Preocupación por perder el trabajo
    "C_006_003_011", # Justicia en la distribución del ingreso

    # --- Percepción política y social ---
    "B_001_101",   # País para todos o para los poderosos
    "B_006_061",   # Aprobación gestión del gobierno
    "A_007_071",   # Escala Izquierda-Derecha
    "A_007_001",   # Interés en política
    "C_001_031",   # Problema más importante del país

    # --- Corrupción y seguridad ---
    "G_002_011",   # Conocimiento directo de acto de corrupción
    "G_005_001",   # Progreso en reducción de corrupción
    "I_001_001",   # Victimización por delito

    # --- Sociodemográfico ---
    "S_001",       # Sexo del entrevistado
    "S_002",       # Edad
    "S_101",       # Nivel educativo (recodificado/armonizado)
    "S_200",       # Estado ocupacional
    "S_301",       # Nivel socioeconómico (apreciación encuestador)
    "S_700",       # Religión
    "S_701",       # Práctica religiosa

    # --- Identificación y ponderación ---
    "X_001",       # País (código numérico idenpa)
    "X_002",       # Año de investigación
    "X_004",       # Región / área geográfica
    "X_008",       # Tamaño de ciudad
    "X_020",       # Factor de ponderación muestral
]

print(f"Total de variables seleccionadas: {len(VARIABLES_SELECCIONADAS)}")
print("  (1 target + 39 features/identificadores/ponderación)")

In [ ]:
# =============================================================================
# Lectura del CSV y construcción del mapeo
# =============================================================================

# cargar el csv de las variables seleccionadas y construir un diccionario de mapeo
df_variables_sel = pd.read_csv(PATHS["FILE_VAR_SELECTION"], sep=";", encoding="utf-8", dtype=str, keep_default_na=False)

def construir_mapeo_variables(ruta_csv: Path, variables_objetivo: list) -> dict:
    """
    Lee latinobarometro_variables.csv (formato largo, sep=';') y construye:

        {
            "A_003_031": {
                "titulo_es": "Satisfacción con la democracia",
                "olas": {
                    "LAT1995": "p21",
                    "LAT1996": "p20",
                    ...
                }
            },
            ...
        }

    Columnas del CSV: year | variable | question
    - variable : código estandarizado (ej. A_003_031)
    - question : nombre de la columna en el .dta de ese año (ej. p21)
    - year     : año numérico (ej. 1995) → se convierte a clave LAT1995
    """
    df = pd.read_csv(ruta_csv, sep=";", encoding="utf-8", dtype=str, keep_default_na=False)

    variables_obj_set = set(variables_objetivo)
    df_filtrado = df[df["variable"].str.strip().isin(variables_obj_set)].copy()

    mapeo = {}
    for codigo_var, grupo in df_filtrado.groupby("variable"):
        olas = {}
        titulo_es = ""
        for _, fila in grupo.iterrows():
            year_str = str(fila["year"]).strip()
            question = str(fila["question"]).strip()
            if year_str and question:
                olas[f"LAT{year_str}"] = question
            if not titulo_es:
                # titulo_es = str(fila["label"]).strip()
                titulo_es = str(df_variables_sel[df_variables_sel["variable"] == codigo_var]["label"].iloc[0]).strip()

        mapeo[codigo_var.strip()] = {
            "titulo_es": titulo_es,
            "olas": olas,
        }

    no_encontradas = variables_obj_set - set(mapeo.keys())
    if no_encontradas:
        print(f"⚠️  Variables no encontradas en el CSV: {sorted(no_encontradas)}")

    return mapeo


# Construir el mapeo
print("Construyendo diccionario de mapeo desde el CSV de variables...")
MAPEO_VARIABLES = construir_mapeo_variables(PATHS["FILE_LB_VAR_MAPPING"], VARIABLES_SELECCIONADAS)

print(f"✓ Mapeo construido para {len(MAPEO_VARIABLES)} variables.")

#print(MAPEO_VARIABLES)

# Inspección de ejemplo
var_ejemplo = "A_003_031"
info_ejemplo = MAPEO_VARIABLES.get(var_ejemplo)
if info_ejemplo:
    print(f"\nEjemplo — {var_ejemplo}:")
    print(f"  Título ES: {info_ejemplo['titulo_es']}")
    print("  Nombres originales por ola (question):")
    for ola, nombre in list(info_ejemplo["olas"].items())[:6]:
        print(f"    {ola}: {nombre}")
    print("    ...")
else:
    print(f"\n⚠️  '{var_ejemplo}' no encontrada en el CSV.")

## 4. Carga y consolidación de archivos Latinobarómetro

Esta sección recorre todos los archivos `.dta` de la carpeta `data/latinobarometro/`,
extrae las columnas correspondientes a las variables seleccionadas para cada ola,
renombra las columnas con los códigos estandarizados y consolida todo en un único
DataFrame. Cada archivo se libera de memoria (`gc.collect()`) tras su procesamiento
para minimizar el consumo de RAM.

In [ ]:
# =============================================================================
# Función auxiliar: inferir la clave de ola desde el nombre del archivo
# =============================================================================

def inferir_clave_ola(nombre_archivo: str) -> str | None:
    """
    Extrae la clave de ola (ej. 'LAT2022') a partir del nombre de un archivo .dta.
    """
    match = re.search(r"(1[9][0-9]{2}|20[0-9]{2})", nombre_archivo)
    return f"LAT{match.group(1)}" if match else None


# Prueba de la función auxiliar
ejemplos = [
    "Latinobarometro_2022.dta",
    "latinobarometro2005_v2.dta",
    "LB_1997.dta",
    "Latinobarometro2020Oct.dta",
    "datos_sin_año.dta",
]
print("Prueba de inferencia de claves de ola:")
for ej in ejemplos:
    print(f"  {ej:45s} → {inferir_clave_ola(ej)}")

In [ ]:
# =============================================================================
# Descubrimiento y validación de archivos .dta
# =============================================================================

# Obtener la lista de archivos .dta disponibles en la carpeta
archivos_dta = sorted(PATHS["FOLDER_RAW_LB"].glob("*.dta"))

if not archivos_dta:
    raise FileNotFoundError(
        f"No se encontraron archivos .dta en {PATHS['FOLDER_RAW_LB']}. "
        "Verificar que los archivos estén en la carpeta correcta."
    )

print(f"Archivos .dta encontrados: {len(archivos_dta)}")
print()

# Verificar que cada archivo tiene una clave de ola reconocida
archivos_validos = []
archivos_sin_clave = []

for archivo in archivos_dta:
    clave = inferir_clave_ola(archivo.name)
    if clave is None:
        archivos_sin_clave.append(archivo.name)
    else:
        archivos_validos.append((archivo, clave))

# Mostrar tabla de archivos válidos
print(f"{'Archivo':<55} {'Clave de ola':<12} {'En mapeo Excel'}")
print("-" * 82)
for archivo, clave in archivos_validos:
    en_mapeo = "✓" if any(
        clave in info["olas"] for info in MAPEO_VARIABLES.values()
    ) else "⚠️ no en mapeo"
    print(f"  {archivo.name:<53} {clave:<12} {en_mapeo}")

if archivos_sin_clave:
    print(f"\n⚠️  Archivos sin año detectado (serán omitidos): {archivos_sin_clave}")

print(f"\n✓ {len(archivos_validos)} archivos listos para procesar.")

In [ ]:
# =============================================================================
# Función principal: procesar un archivo .dta y extraer variables
# =============================================================================

def _detectar_encoding(ruta_archivo: Path) -> dict:
    """
    Prueba encodings en orden hasta encontrar uno que lea los metadatos sin error.
    Devuelve el kwarg a pasar a read_dta (ej. {} o {"encoding": "latin1"}).
    """
    for enc in (None, "latin1"):
        kwargs = {} if enc is None else {"encoding": enc}
        try:
            pyreadstat.read_dta(str(ruta_archivo), metadataonly=True, **kwargs)
            return kwargs
        except Exception:
            continue
    raise IOError(f"No se pudo determinar el encoding de {ruta_archivo.name}")


def procesar_ola_dta(
    ruta_archivo: Path,
    clave_ola: str,
    mapeo_variables: dict,
) -> pd.DataFrame:
    """
    Lee solo las columnas necesarias de un archivo .dta de Latinobarómetro:

    1. Detecta el encoding leyendo solo metadatos (sin cargar datos).
    2. Resuelve los nombres de columna de forma case-insensitive.
    3. Lee únicamente las columnas requeridas con usecols.

    Las variables sin correspondencia en esta ola se agregan como NaN.
    """
    # -------------------------------------------------------------------------
    # Paso 1: Determinar qué columnas (question) corresponden a esta ola
    # -------------------------------------------------------------------------
    rename_map = {}       # nombre_en_dta (original del CSV) → código estandarizado
    variables_ausentes = []

    for codigo_std, info in mapeo_variables.items():
        if clave_ola in info["olas"]:
            rename_map[info["olas"][clave_ola]] = codigo_std
        else:
            variables_ausentes.append(codigo_std)

    # -------------------------------------------------------------------------
    # Paso 2: Leer metadatos para obtener nombres reales de columna (sin datos)
    # -------------------------------------------------------------------------
    enc_kwargs = _detectar_encoding(ruta_archivo)
    _, meta = pyreadstat.read_dta(str(ruta_archivo), metadataonly=True, **enc_kwargs)

    # Lista de columnas
    #columnas_dta = meta.column_names

    # Guardar en CSV
    #pd.DataFrame({"columna": columnas_dta}).to_csv(
    #    "..\data\\columnas_dta.csv",
    #    index=False,
    #    encoding="utf-8-sig"
    #)

    # -------------------------------------------------------------------------
    # Paso 3: Resolver nombres reales (case-insensitive)
    # -------------------------------------------------------------------------
    cols_lower_a_real = {col.lower(): col for col in meta.column_names}

    rename_map_resuelto = {}   # nombre_real_en_dta → código estandarizado
    variables_no_en_dta = []

    for nombre_csv, codigo_std in rename_map.items():
        nombre_real = cols_lower_a_real.get(nombre_csv.lower())
        if nombre_real:
            rename_map_resuelto[nombre_real] = codigo_std
        else:
            variables_no_en_dta.append((nombre_csv, codigo_std))
            variables_ausentes.append(codigo_std)

    # -------------------------------------------------------------------------
    # Paso 4: Leer SOLO las columnas necesarias
    # -------------------------------------------------------------------------
    columnas_a_leer = list(rename_map_resuelto.keys())
    df_raw, _ = pyreadstat.read_dta(
        str(ruta_archivo),
        usecols=columnas_a_leer,
        **enc_kwargs,
    )

    # -------------------------------------------------------------------------
    # Paso 5: Renombrar, agregar ausentes y ordenar
    # -------------------------------------------------------------------------
    df_ola = df_raw.rename(columns=rename_map_resuelto)
    del df_raw
    gc.collect()

    for codigo_std in variables_ausentes:
        if codigo_std not in df_ola.columns:
            df_ola[codigo_std] = np.nan

    df_ola.insert(0, "ola", clave_ola)
    cols_ord = ["ola"] + sorted(c for c in df_ola.columns if c != "ola")
    df_ola = df_ola[cols_ord]

    if variables_no_en_dta:
        print(f"Total de variables no encontradas en el .dta: {len(variables_no_en_dta)}")
        for nombre_orig, codigo_std in variables_no_en_dta:
            print(f"    ⚠️  [{clave_ola}] '{nombre_orig}' (→{codigo_std}) no encontrada en el .dta.")

    return df_ola


print("✓ Función 'procesar_ola_dta' definida correctamente.")

In [ ]:
# =============================================================================
# Prueba de procesamiento con el primer archivo (verificación previa)
# =============================================================================
# Se prueba el procesamiento con el primer archivo antes de ejecutar el ciclo
# completo, para detectar problemas de formato o codificación de forma temprana.

archivo_prueba, clave_prueba = archivos_validos[22]
print(f"Probando con: {archivo_prueba.name} ({clave_prueba})")
print("-" * 60)

df_prueba = procesar_ola_dta(archivo_prueba, clave_prueba, MAPEO_VARIABLES)

print(f"\n  Filas     : {len(df_prueba):,}")
print(f"  Columnas  : {len(df_prueba.columns)}")
print(f"  Columnas con todos NaN en esta ola:")
cols_nan = [c for c in df_prueba.columns if df_prueba[c].isna().all()]
for c in cols_nan:
    print(f"    - {c} (variable ausente en {clave_prueba})")

print("\nPrimeras 3 filas de la prueba:")
df_prueba.head(3)

In [ ]:
# =============================================================================
# Ciclo principal: procesamiento de todas las olas
# =============================================================================
# Estrategia de memoria:
#   1. Procesar un archivo .dta a la vez.
#   2. Al terminar cada archivo, el DataFrame completo se descarta (del df_raw)
#      dentro de `procesar_ola_dta` y gc.collect() libera la memoria.
#   3. Solo el DataFrame reducido (variables seleccionadas) se acumula en `bloques`.
#   4. Al final, pd.concat une todos los bloques en un único DataFrame consolidado.

bloques = []  # Lista de DataFrames, uno por ola
log_procesamiento = []  # Registro para el reporte final

print("Iniciando procesamiento de olas de Latinobarómetro...")
print("=" * 70)

for archivo, clave_ola in tqdm(archivos_validos, desc="Procesando olas"):
    try:
        print(f"\nProcesando {clave_ola} ({archivo.name})...")
        df_ola = procesar_ola_dta(archivo, clave_ola, MAPEO_VARIABLES)

        n_filas = len(df_ola)
        n_vars_con_datos = sum(
            1 for c in df_ola.columns
            if c not in ("ola",) and not df_ola[c].isna().all()
        )
        n_vars_ausentes = sum(
            1 for c in df_ola.columns
            if c not in ("ola",) and df_ola[c].isna().all()
        )

        bloques.append(df_ola)
        log_procesamiento.append({
            "ola"         : clave_ola,
            "archivo"     : archivo.name,
            "n_registros" : n_filas,
            "vars_con_datos": n_vars_con_datos,
            "vars_ausentes" : n_vars_ausentes,
            "estado"      : "OK",
        })

        # Liberar el DataFrame de la ola de la variable local
        # (el objeto aún vive en `bloques`, pero liberamos la referencia local)
        del df_ola
        gc.collect()

    except Exception as e:
        print(f"\n  ✗ Error en {clave_ola} ({archivo.name}): {e}")
        log_procesamiento.append({
            "ola"         : clave_ola,
            "archivo"     : archivo.name,
            "n_registros" : 0,
            "vars_con_datos": 0,
            "vars_ausentes" : 0,
            "estado"      : f"ERROR: {e}",
        })

print("\n✓ Procesamiento de archivos completado.")
print(f"  Olas procesadas exitosamente: {sum(1 for r in log_procesamiento if r['estado'] == 'OK')}")
print(f"  Olas con error              : {sum(1 for r in log_procesamiento if 'ERROR' in r['estado'])}")

In [ ]:
# =============================================================================
# Reporte de procesamiento por ola
# =============================================================================

df_log = pd.DataFrame(log_procesamiento)
print("Reporte de procesamiento por ola:")
print("=" * 85)
print(f"  {'Ola':<10} {'Registros':>10} {'Vars con datos':>15} {'Vars ausentes':>14} {'Estado'}")
print("-" * 85)
for _, fila in df_log.iterrows():
    icono = "✓" if fila["estado"] == "OK" else "✗"
    print(
        f"  {icono} {fila['ola']:<9} "
        f"{fila['n_registros']:>10,} "
        f"{fila['vars_con_datos']:>15} "
        f"{fila['vars_ausentes']:>14} "
        f"{fila['estado']}"
    )

print("-" * 85)
print(f"  {'TOTAL':<10} {df_log['n_registros'].sum():>10,}")

In [ ]:
# =============================================================================
# Consolidación de todos los bloques en un único DataFrame
# =============================================================================

print("Consolidando todos los bloques en un único DataFrame...")

df_lb = pd.concat(bloques, axis=0, ignore_index=True, sort=False)

# Liberar la lista de bloques
del bloques
gc.collect()

print(f"✓ DataFrame consolidado:")
print(f"  Dimensiones  : {df_lb.shape[0]:,} filas × {df_lb.shape[1]} columnas")
print(f"  Memoria aprox: {df_lb.memory_usage(deep=True).sum() / 1_048_576:.1f} MB")
print(f"  Columnas     : {list(df_lb.columns)}")

## 5. Extracción del listado de países del estudio

A partir del DataFrame consolidado de Latinobarómetro se obtiene el listado
de países únicos. Este listado se utilizará posteriormente para filtrar
los indicadores de V-Dem a los países relevantes para el estudio.

In [ ]:
# =============================================================================
# Extracción del listado de países y mapeo a ISO3
# =============================================================================

# Mapeo de códigos numéricos de Latinobarómetro (idenpa) a nombres e ISO3
# Fuente: documentación oficial de Latinobarómetro
MAPEO_PAISES_LB = {
    32  : {"nombre": "Argentina",            "iso3": "ARG"},
    68  : {"nombre": "Bolivia",              "iso3": "BOL"},
    76  : {"nombre": "Brasil",               "iso3": "BRA"},
    152 : {"nombre": "Chile",                "iso3": "CHL"},
    170 : {"nombre": "Colombia",             "iso3": "COL"},
    188 : {"nombre": "Costa Rica",           "iso3": "CRI"},
    214 : {"nombre": "República Dominicana", "iso3": "DOM"},
    218 : {"nombre": "Ecuador",              "iso3": "ECU"},
    222 : {"nombre": "El Salvador",          "iso3": "SLV"},
    320 : {"nombre": "Guatemala",            "iso3": "GTM"},
    340 : {"nombre": "Honduras",             "iso3": "HND"},
    484 : {"nombre": "México",               "iso3": "MEX"},
    558 : {"nombre": "Nicaragua",            "iso3": "NIC"},
    591 : {"nombre": "Panamá",               "iso3": "PAN"},
    600 : {"nombre": "Paraguay",             "iso3": "PRY"},
    604 : {"nombre": "Perú",                 "iso3": "PER"},
    858 : {"nombre": "Uruguay",              "iso3": "URY"},
    862 : {"nombre": "Venezuela",            "iso3": "VEN"},
}

# Obtener códigos numéricos únicos presentes en los datos
# X_001 contiene el código numérico del país (idenpa)
codigos_en_datos = (
    df_lb["X_001"]
    .dropna()
    .astype(float)
    .astype(int)
    .unique()
)
codigos_en_datos = sorted(codigos_en_datos)

# Construir el DataFrame de países del estudio
registros_paises = []
for codigo in codigos_en_datos:
    if codigo in MAPEO_PAISES_LB:
        info = MAPEO_PAISES_LB[codigo]
        registros_paises.append({
            "idenpa": codigo,
            "nombre": info["nombre"],
            "iso3"  : info["iso3"],
        })
    else:
        print(f"⚠️  Código de país no reconocido en el mapeo: {codigo}")
        registros_paises.append({
            "idenpa": codigo,
            "nombre": f"Desconocido_{codigo}",
            "iso3"  : "UNK",
        })

df_paises = pd.DataFrame(registros_paises)
LISTA_ISO3_ESTUDIO = df_paises["iso3"].tolist()

print(f"Países identificados en los datos de Latinobarómetro: {len(df_paises)}")
print()
print(df_paises.to_string(index=False))
print(f"\nCódigos ISO3 para filtrado V-Dem:")
print(f"  {LISTA_ISO3_ESTUDIO}")

In [ ]:
# =============================================================================
# Agregar columnas de nombre de país e ISO3 al DataFrame consolidado
# =============================================================================

# Crear mapeo: idenpa (float) → nombre e iso3
mapa_idenpa_a_nombre = {float(k): v["nombre"] for k, v in MAPEO_PAISES_LB.items()}
mapa_idenpa_a_iso3   = {float(k): v["iso3"]   for k, v in MAPEO_PAISES_LB.items()}

df_lb["pais_nombre"] = df_lb["X_001"].map(mapa_idenpa_a_nombre)
df_lb["pais_iso3"]   = df_lb["X_001"].map(mapa_idenpa_a_iso3)

# Verificar cobertura del mapeo
n_sin_nombre = df_lb["pais_nombre"].isna().sum()
n_sin_iso3   = df_lb["pais_iso3"].isna().sum()

print(f"Columnas 'pais_nombre' e 'pais_iso3' agregadas al DataFrame.")
print(f"  Registros sin nombre mapeado : {n_sin_nombre:,}")
print(f"  Registros sin ISO3 mapeado   : {n_sin_iso3:,}")

if n_sin_nombre > 0:
    codigos_no_mapeados = (
        df_lb[df_lb["pais_nombre"].isna()]["X_001"]
        .dropna().unique()
    )
    print(f"  ⚠️  Códigos sin mapear: {codigos_no_mapeados}")

## 6. Exportación del DataFrame de Latinobarómetro

In [ ]:
# =============================================================================
# Exportación del DataFrame consolidado a CSV
# =============================================================================

print(f"Exportando DataFrame de Latinobarómetro a: {PATHS["FILE_BASE_LB"]}")

# Crear la carpeta de destino si no existe
PATHS["FILE_BASE_LB"].parent.mkdir(parents=True, exist_ok=True)

df_lb.to_csv(
    PATHS["FILE_BASE_LB"],
    index=False,
    encoding="utf-8-sig",  # UTF-8 con BOM para compatibilidad con Excel
)

# Verificación del archivo generado
tamanio_mb = PATHS["FILE_BASE_LB"].stat().st_size / 1_048_576

print(f"✓ Archivo generado exitosamente.")
print(f"  Ruta    : {PATHS['FILE_BASE_LB']}")
print(f"  Tamaño  : {tamanio_mb:.1f} MB")
print(f"  Filas   : {len(df_lb):,}")
print(f"  Columnas: {len(df_lb.columns)}")

# Verificación de integridad: releer el CSV y comparar dimensiones
df_verificacion = pd.read_csv(PATHS["FILE_BASE_LB"], nrows=5)
print(f"\nVerificación de lectura (primeras 5 filas):")
print(f"  Columnas leídas: {len(df_verificacion.columns)} ← debe coincidir con {len(df_lb.columns)}")
del df_verificacion

## 7. Carga y filtrado de V-Dem

Se lee el archivo `V-Dem-CY-Core-v16.csv`, se filtran los registros
correspondientes a los países del estudio y al período 1995–2024,
y se seleccionan las columnas definidas en la metodología.

**Nota sobre versiones de variables:** Se utilizan exclusivamente las columnas
base sin sufijo (ej. `v2x_polyarchy`, NO `v2x_polyarchy_osp`), que corresponden
a las estimaciones del modelo de medición en escala de intervalo continua.

**Nota sobre dirección de índices:** Algunos índices tienen dirección invertida
(mayor valor = peor situación democrática). Estas variables están documentadas
en `variables_seleccionadas_metodologia.md` y se señalan en el reporte de
esta sección.

In [ ]:
# =============================================================================
# Definición de columnas V-Dem a seleccionar
# =============================================================================

# Columnas de identificación (siempre incluir)
COLS_ID_VDEM = [
    "country_name",       # Nombre del país en inglés
    "country_text_id",    # Código ISO3 del país
    "country_id",         # ID numérico V-Dem
    "year",               # Año
    "COWcode",            # Correlates of War code (útil para merge con otras fuentes)
]

# Índices de alto nivel — SOLO para Regresión Logística Ordinal (baseline)
COLS_HIGHLEVEL_VDEM = [
    "v2x_polyarchy",      # Democracia electoral (poliarquía)
    "v2x_libdem",         # Democracia liberal
    "v2x_partipdem",      # Democracia participativa
    "v2x_delibdem",       # Democracia deliberativa
    "v2x_egaldem",        # Democracia igualitaria
]

# Índices mid-level y especializados — Para XGBoost, CatBoost, LightGBM, TabNet
COLS_MIDLEVEL_VDEM = [
    # Componentes liberales (subcomponentes de v2x_libdem)
    "v2xcl_rol",          # Igualdad ante la ley y libertades individuales
    "v2x_jucon",          # Restricciones judiciales al ejecutivo
    "v2xlg_legcon",       # Restricciones legislativas al ejecutivo

    # Libertad de expresión (subcomponente de v2x_polyarchy)
    "v2x_freexp_altinf",  # Libertad de expresión e información alternativa

    # Corrupción (dirección original: mayor valor = MÁS corrupción)
    "v2x_corr",           # Corrupción política (índice global)
    "v2x_execorr",        # Corrupción ejecutiva
    "v2x_pubcorr",        # Corrupción sector público

    # Neopatrimonialismo (dirección original: mayor valor = PEOR)
    "v2x_neopat",         # Neopatrimonialismo
    "v2xnp_regcorr",      # Corrupción del régimen

    # Accountability e instituciones
    "v2x_accountability_osp",  # Accountability (versión OSP [0,1]; la base es z-score)
    "v2xcs_ccsi",         # Índice de sociedad civil
    "v2x_rule",           # Estado de derecho
    "v2x_cspart",         # Participación de la sociedad civil

    # Igualdad y exclusión
    "v2x_egal",           # Componente igualitario
    "v2xeg_eqdr",         # Distribución igualitaria de recursos

    # Exclusión (dirección original: mayor valor = MÁS exclusión)
    "v2xpe_exlsocgr",     # Exclusión por grupo social
    "v2xpe_exlecon",      # Exclusión por posición económica

    # Género
    "v2x_gender",         # Empoderamiento político de mujeres
]

# Lista completa de columnas a retener
COLS_VDEM_TOTAL = COLS_ID_VDEM + COLS_HIGHLEVEL_VDEM + COLS_MIDLEVEL_VDEM

print(f"Columnas V-Dem a seleccionar: {len(COLS_VDEM_TOTAL)}")
print(f"  Identificación     : {len(COLS_ID_VDEM)}")
print(f"  High-level indices : {len(COLS_HIGHLEVEL_VDEM)}")
print(f"  Mid-level indices  : {len(COLS_MIDLEVEL_VDEM)}")
print("  (Nota: la inversión de escalas se aplica en el notebook 2 de preprocesamiento)")

In [ ]:
# =============================================================================
# Lectura del CSV de V-Dem
# =============================================================================

print(f"Leyendo V-Dem CY-Core desde: {PATHS["FILE_RAW_VDEM"]}")
print("(Se leen solo las columnas seleccionadas para optimizar memoria)")
print()

# Leer solo los encabezados (nrows=0) para verificar qué columnas existen
# Pandas maneja correctamente las comillas en el CSV
cols_disponibles_vdem = set(pd.read_csv(PATHS["FILE_RAW_VDEM"], nrows=0, encoding="utf-8").columns)

cols_a_leer       = [c for c in COLS_VDEM_TOTAL if c in cols_disponibles_vdem]
cols_ausentes_vdem = [c for c in COLS_VDEM_TOTAL if c not in cols_disponibles_vdem]

print(f"Columnas solicitadas: {len(COLS_VDEM_TOTAL)}")
print(f"Columnas encontradas: {len(cols_a_leer)}")
print(f"Columnas no encontradas: {len(cols_ausentes_vdem)}")
if cols_ausentes_vdem:
    print(f"⚠️  Columnas seleccionadas NO encontradas en el CSV V-Dem:")
    for c in cols_ausentes_vdem:
        print(f"    - {c}")
    print()

# Leer solo las columnas necesarias
df_vdem_completo = pd.read_csv(
    PATHS["FILE_RAW_VDEM"],
    usecols=cols_a_leer,
    low_memory=False,
    encoding="utf-8",
)

print(f"✓ V-Dem leído correctamente.")
print(f"  Columnas leídas  : {df_vdem_completo.shape[1]} de {len(COLS_VDEM_TOTAL)} solicitadas")
print(f"  Dimensiones      : {df_vdem_completo.shape[0]:,} filas × {df_vdem_completo.shape[1]} columnas")
print(f"  Rango de años    : {df_vdem_completo['year'].min()} – {df_vdem_completo['year'].max()}")
print(f"  Países totales   : {df_vdem_completo['country_name'].nunique()}")

In [ ]:
# =============================================================================
# Filtrado por países del estudio y período 1995–2024
# =============================================================================

# Verificar qué ISO3 del estudio están en V-Dem
iso3_en_vdem     = set(df_vdem_completo["country_text_id"].unique())
iso3_encontrados = [iso3 for iso3 in LISTA_ISO3_ESTUDIO if iso3 in iso3_en_vdem]
iso3_no_en_vdem  = [iso3 for iso3 in LISTA_ISO3_ESTUDIO if iso3 not in iso3_en_vdem]

if iso3_no_en_vdem:
    print(f"⚠️  Países del estudio no encontrados en V-Dem: {iso3_no_en_vdem}")

# Aplicar filtro
mascara_paises = df_vdem_completo["country_text_id"].isin(iso3_encontrados)
mascara_anios  = (df_vdem_completo["year"] >= PARAMETERS["YEAR_START"]) & (df_vdem_completo["year"] <= PARAMETERS["YEAR_END"])

df_vdem = df_vdem_completo[mascara_paises & mascara_anios].copy()

# Liberar el DataFrame completo
del df_vdem_completo
gc.collect()

# Ordenar por país y año
df_vdem = df_vdem.sort_values(["country_text_id", "year"]).reset_index(drop=True)

print(f"Filtro aplicado: países del estudio ({len(iso3_encontrados)}) + años {PARAMETERS['YEAR_START']}–{PARAMETERS['YEAR_END']}")
print(f"\n✓ DataFrame V-Dem filtrado:")
print(f"  Dimensiones  : {df_vdem.shape[0]:,} filas × {df_vdem.shape[1]} columnas")
print(f"  Países       : {df_vdem['country_name'].nunique()}")
print(f"  Rango de años: {df_vdem['year'].min()} – {df_vdem['year'].max()}")

print(f"\nCobertura por país (filas = años disponibles):")
cobertura = (
    df_vdem.groupby(["country_text_id", "country_name"])
    .size()
    .reset_index(name="n_años")
    .sort_values("country_text_id")
)
print(cobertura.to_string(index=False))

In [ ]:
# =============================================================================
# Verificación de cobertura de variables V-Dem seleccionadas
# =============================================================================

print("Cobertura de variables V-Dem en el dataset filtrado:")
print(f"  (países AL, {PARAMETERS['YEAR_START']}–{PARAMETERS['YEAR_END']}, {len(df_vdem)} registros totales)")
print()
print(f"  {'Variable':<30} {'% completitud':>14} {'Min':>8} {'Max':>8}")
print("-" * 67)

for col in COLS_MIDLEVEL_VDEM + COLS_HIGHLEVEL_VDEM:
    if col not in df_vdem.columns:
        print(f"  {col:<30} {'NO DISPONIBLE':>14}")
        continue
    completitud = (1 - df_vdem[col].isna().mean()) * 100
    val_min = df_vdem[col].min() if completitud > 0 else np.nan
    val_max = df_vdem[col].max() if completitud > 0 else np.nan
    print(
        f"  {col:<30} {completitud:>13.1f}% "
        f"{val_min:>8.3f} {val_max:>8.3f}"
    )

## 8. Exportación del DataFrame de V-Dem

In [ ]:
# =============================================================================
# Exportación del DataFrame de V-Dem a CSV
# =============================================================================

print(f"Exportando DataFrame de V-Dem a: {PATHS["FILE_BASE_VDEM"]}")

PATHS["FILE_BASE_VDEM"].parent.mkdir(parents=True, exist_ok=True)

df_vdem.to_csv(
    PATHS["FILE_BASE_VDEM"],
    index=False,
    encoding="utf-8-sig",
    float_format="%.6f",  # 6 decimales para preservar precisión de los índices
)

tamanio_mb_vdem = PATHS["FILE_BASE_VDEM"].stat().st_size / 1_048_576

print(f"✓ Archivo generado exitosamente.")
print(f"  Ruta    : {PATHS['FILE_BASE_VDEM']}")
print(f"  Tamaño  : {tamanio_mb_vdem:.1f} MB")
print(f"  Filas   : {len(df_vdem):,}")
print(f"  Columnas: {len(df_vdem.columns)}")

# Verificación de integridad
df_verif_vdem = pd.read_csv(PATHS["FILE_BASE_VDEM"], nrows=3)
print(f"\nVerificación de lectura (primeras 3 filas):")
print(f"  Columnas leídas: {len(df_verif_vdem.columns)} ← debe coincidir con {len(df_vdem.columns)}")
del df_verif_vdem

## 9. Resumen ejecutivo del pipeline de carga

In [ ]:
# =============================================================================
# Resumen ejecutivo y estadísticas finales
# =============================================================================

print("=" * 70)
print("  RESUMEN EJECUTIVO — PIPELINE DE CARGA DE DATOS")
print("=" * 70)

print("\n📁  LATINOBARÓMETRO")
print(f"    Archivo de salida  : {PATHS['FILE_BASE_LB'].name}")
print(f"    Total registros    : {len(df_lb):,}")
print(f"    Total columnas     : {len(df_lb.columns)}")
print(f"    Olas incluidas     : {df_lb['ola'].nunique()}")
print(f"    Países incluidos   : {df_lb['pais_iso3'].nunique()}")
print(f"    Rango temporal     : {df_lb['ola'].min()} – {df_lb['ola'].max()}")

# Distribución de registros por ola
print("\n    Registros por ola:")
dist_olas = df_lb.groupby("ola").size().reset_index(name="n")
for _, row in dist_olas.iterrows():
    print(f"      {row['ola']}: {row['n']:,}")

print("\n📁  V-DEM")
print(f"    Archivo de salida  : {PATHS['FILE_BASE_VDEM'].name}")
print(f"    Total registros    : {len(df_vdem):,}")
print(f"    Total columnas     : {len(df_vdem.columns)}")
print(f"    Países incluidos   : {df_vdem['country_name'].nunique()}")
print(f"    Rango temporal     : {df_vdem['year'].min()} – {df_vdem['year'].max()}")

print("\n✅  Pipeline completado. Archivos listos para el siguiente notebook:")
print(f"    - {PATHS['FILE_BASE_LB']}")
print(f"    - {PATHS['FILE_BASE_VDEM']}")
print()
print("📝  Próximo paso: 02_preprocesamiento.ipynb")
print("    - Merge Latinobarómetro × V-Dem por (pais_iso3, año)")
print("    - Inversión de escalas (confianza institucional, índices V-Dem invertidos)")
print("    - Imputación MICE para variables con missingness estructural")
print("    - Partición Time-Based Split (Train: 1995–2018 / Test: 2019–2024)")
print("    - Tratamiento especial de Venezuela (solo train hasta 2017)")
print("=" * 70)

## 9. Archivos Adicionales

Se obtiene el conteo de valores por cada variable y ola para análisis de los datos.

In [ ]:
## Conteo de valores por variable y ola

resultado = []

# Recorrer todas las columnas excepto "ola"
for col in df_lb.columns:
    if col == "ola":
        continue

    conteo = (
        df_lb
        .groupby("ola")[col]
        .value_counts(dropna=False)
        .reset_index(name="total_apariciones")
    )

    conteo.insert(1, "columna", col)
    conteo.rename(columns={col: "valor"}, inplace=True)

    resultado.append(conteo)

# Unir todos los resultados
df_resultado = pd.concat(resultado, ignore_index=True)

# Exportar
df_resultado.to_csv(
    PATHS["FILE_FREQUENCY_TABLE"],
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
df_resultado.info()

Se obtiene una muestra aleatoria del Latinobarómetro de cada país por cada año, esto para ajustar los siguientes notebooks previo al procesamiento del archivo base.

In [ ]:
# Obtener una muestra aleatoria del LB de 18 a 25 registros por país y ola para inspección manual

# Semilla para que el resultado sea reproducible (opcional)
rng = np.random.default_rng(seed=PARAMETERS["SEED"])

muestras = []

# groups es un diccionario:
# {(ola, pais_iso3): índices}
for _, idx in df_lb.groupby(["ola", "pais_iso3"]).groups.items():

    grupo = df_lb.loc[idx]

    n = min(rng.integers(18, 26), len(grupo))

    muestras.append(
        grupo.sample(
            n=n,
            random_state=rng.integers(0, 2**32 - 1)
        )
    )

df_muestra = pd.concat(muestras, ignore_index=True)

print(df_muestra.columns.tolist())   # <-- verifica aquí


# Exportar
df_muestra.to_csv(
    PATHS["FILE_BASE_LB_SAMPLE"],
    index=False,
    encoding="utf-8-sig"
)

print(f"Registros originales: {len(df_lb):,}")
print(f"Registros en la muestra: {len(df_muestra):,}")

In [ ]:
df_muestra.info()